# Notebook 07 – Adaptive Profile Repository (Two Repositories)

Convert Notebook 06's scored outputs into the website-ready knowledge base. The LLM **never** decides UI adaptations — it only renders the selected base profile and applies the applicable trait nudges.

**Inputs (NB06):**
- `reports/AdaptiveProfiles/merged_base_profiles.xlsx`
- `reports/AdaptiveProfiles/scored_trait_modifiers.csv`

**Outputs:**
- `data/outputs/candidate_base_profile.json` — base UI configurations (Persona + Mood + Device)
- `data/outputs/trait_modifiers.json` — Big Five ordinal nudges (nested by trait → level)
- `data/outputs/profile_lookup.json` — persona → mood → device → profile_id index
- `reports/AdaptiveProfileRepository/base_profile_catalog.xlsx`
- `reports/AdaptiveProfileRepository/repository_summary.md`

**Conflict rule:** the base profile takes precedence; nudges only adjust configurable properties within limits or fill unspecified values.

In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.evidence_engine.repository_exporter import run_repository_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("AdaptiveProfileRepository")
OUTPUT_DIR = PATHS.data_outputs

print(f"JSON output: {OUTPUT_DIR}")
print(f"Catalog output: {REPORTS}")

JSON output: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs
Catalog output: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AdaptiveProfileRepository


## Check Upstream Inputs

In [2]:
INPUTS = {
    "Merged base profiles (NB06)": PATHS.reports / "AdaptiveProfiles" / "merged_base_profiles.xlsx",
    "Scored trait modifiers (NB06)": PATHS.reports / "AdaptiveProfiles" / "scored_trait_modifiers.csv",
}
for label, path in INPUTS.items():
    fallback = path.with_suffix(".csv" if path.suffix == ".xlsx" else ".xlsx")
    status = "OK" if path.exists() else ("OK (fallback)" if fallback.exists() else "MISSING — run NB06")
    print(f"{label}: {status}")

Merged base profiles (NB06): OK
Scored trait modifiers (NB06): OK


## Export Both Repositories

In [3]:
result = run_repository_pipeline(PROJECT_ROOT, OUTPUT_DIR, REPORTS)

base_profiles = result.base_profiles
trait_modifiers = result.trait_modifiers
validation = result.validation
summary = result.summary

print(f"Base profiles exported: {summary['base_profiles_exported']}")
print(f"Trait modifiers exported: {summary['trait_modifiers_exported']}")
print(f"Validation: {'PASS' if validation.is_valid else 'FAIL'}")
if not validation.is_valid:
    for issue in validation.issues[:10]:
        print(f"  - {issue}")

INFO: Exported two-repository knowledge base to /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs


Base profiles exported: 26
Trait modifiers exported: 17
Validation: PASS


## Sample Base Profile (`candidate_base_profile.json`)

In [4]:
print(json.dumps(base_profiles[0], indent=2, ensure_ascii=False))

{
  "profile_id": "base_001",
  "context": {
    "persona": null,
    "mood": "Excited",
    "device": null
  },
  "ui_profile": {
    "sticky_header": "Yes"
  },
  "evidence": {
    "profile_score": 0.708922,
    "strength": "Strong",
    "participants": 10,
    "average_support": 0.05,
    "average_confidence": 1.0,
    "average_lift": 1.550388,
    "statistical_score": 1.0,
    "feature_importance": 0.979062,
    "shap_score": 0.0
  },
  "metadata": {
    "merged_count": 1,
    "num_ui_adaptations": 1,
    "created_from": "Adaptive Builder",
    "version": "2.0"
  }
}


## Trait Modifiers (`trait_modifiers.json`)

Nested by trait → level. Each nudge keeps its provenance and evidence score.

In [5]:
sample_trait = next(iter(trait_modifiers["modifiers"]))
print(f"metadata: {json.dumps(trait_modifiers['metadata'])}")
print(f"\nExample trait — {sample_trait}:")
print(json.dumps(trait_modifiers["modifiers"][sample_trait], indent=2, ensure_ascii=False))

metadata: {"total_modifiers": 17, "data_driven": 5, "theory": 12}

Example trait — Extraversion:
{
  "Low": [
    {
      "property": "recommendation_emphasis",
      "nudge": -1,
      "direction": "decrease",
      "provenance": "theory",
      "evidence_score": 0.323285,
      "strength": "Moderate"
    }
  ],
  "High": [
    {
      "property": "recommendation_emphasis",
      "nudge": -1,
      "direction": "decrease",
      "provenance": "data-driven",
      "evidence_score": 0.635308,
      "strength": "Strong"
    },
    {
      "property": "information_density",
      "nudge": -1,
      "direction": "decrease",
      "provenance": "data-driven",
      "evidence_score": 0.5782,
      "strength": "Strong"
    },
    {
      "property": "animation_level",
      "nudge": 1,
      "direction": "increase",
      "provenance": "theory",
      "evidence_score": 0.01305,
      "strength": "Weak"
    }
  ]
}


## Profile Lookup Index (`profile_lookup.json`)

persona → mood → device → profile_id

In [6]:
lookup_sample = {key: result.lookup[key] for key in list(result.lookup.keys())[:3]}
print(json.dumps(lookup_sample, indent=2, ensure_ascii=False))

{
  "_any": {
    "Excited": {
      "_any": "base_001"
    },
    "Happy": {
      "Smartphone": "base_004",
      "_any": "base_005"
    },
    "Neutral": {
      "Laptop/Desktop": "base_009",
      "_any": "base_019"
    },
    "Stressed": {
      "_any": "base_011"
    },
    "Relaxed": {
      "_any": "base_012"
    },
    "Bored": {
      "_any": "base_013"
    },
    "_any": {
      "Laptop/Desktop": "base_025",
      "Smartphone": "base_026"
    }
  },
  "Researcher": {
    "Relaxed": {
      "Smartphone": "base_002",
      "_any": "base_003"
    },
    "_any": {
      "Laptop/Desktop": "base_008",
      "_any": "base_022"
    },
    "Neutral": {
      "Smartphone": "base_014",
      "_any": "base_020"
    }
  },
  "Deal Hunter": {
    "_any": {
      "_any": "base_006"
    }
  }
}


## Exports

In [7]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")

Export locations:
- candidate_base_profile_json: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs/candidate_base_profile.json
- trait_modifiers_json: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs/trait_modifiers.json
- profile_lookup_json: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs/profile_lookup.json
- base_profile_catalog_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AdaptiveProfileRepository/base_profile_catalog.xlsx
- repository_summary_md: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AdaptiveProfileRepository/repository_summary.md
